# Homework 2

This notebook contains the code and answer(s) for Problem 1. of Homework 2.

---

Problem 1.) Using data from college football seasons 2022 to 2025, filter to the games that are only FBS v FBS and make the plots from the Expected Points notebook from class using that data.  
Explain in a paragraph the differences between the NFL model and the FBS-only model.  

In [8]:
# Import the neccesary libraries
import os
import tabulate
import requests
import pandas as pd
import numpy as np
import pyarrow
import sportsdataverse as sdv
import polars as pl
from great_tables import GT, md
#from adjustText import adjust_text
#import matplotlib as plt
#import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

In [9]:
# Import college football data from 2022 to 2025
cfb_data = sdv.cfb.load_cfb_pbp([2022,2023,2024,2025], return_as_pandas=False)

# Convert the polar files to pandas dataframes
cfb_data = cfb_data.to_pandas(use_pyarrow_extension_array=False)

With the data, let's filter it to only incude FBS vs FBS schools.

In [10]:
# Examine the available columns of the play by play data
col_list = cfb_data.columns.tolist()
pd.set_option('display.max_seq_items', None)
print("\n".join(cfb_data.columns))

season
game_id
game_play_number
pos_team_id
pos_team
def_pos_team_id
def_pos_team
pos_team_score
def_pos_team_score
half
period
down
distance
EPA
wpa
wp_before
wp_after
def_wp_before
def_wp_after
penalty_detail
yds_penalty
penalty_1st_conv
def_EPA
rz_play
scoring_opp
middle_8
stuffed_run
change_of_pos_team
downs_turnover
pos_score_diff_start
pos_score_pts
home_wp_before
away_wp_before
home_wp_after
away_wp_after
end_of_half
lead_pos_team
lead_play_type
lag_pos_team
orig_play_type
offense_score_play
defense_score_play
pos_score_diff
change_of_poss
rusher_player_name
yds_rushed
passer_player_name
receiver_player_name
yds_receiving
yds_sacked
sack_players
sack_player_name
sack_player_name2
pass_breakup_player_name
interception_player_name
yds_int_return
fumble_player_name
fumble_forced_player_name
fumble_recovered_player_name
yds_fumble_return
punter_player_name
yds_punted
yds_punt_return
yds_punt_gained
punt_block_player_name
punt_block_return_player_name
fg_kicker_player_name
yds_fg
fg_

In [11]:
# Create a list of all FBS schools
fbs_teams = ["Boston College", "California", "Clemson", "Duke", "Florida State",
    "Georgia Tech", "Louisville", "Miami", "NC State", "North Carolina",
    "Pittsburgh", "SMU", "Stanford", "Syracuse", "Virginia",
    "Virginia Tech", "Wake Forest", "Illinois", "Indiana", "Iowa", "Maryland", "Michigan",
    "Michigan State", "Minnesota", "Nebraska", "Northwestern", "Ohio State",
    "Oregon", "Penn State", "Purdue", "Rutgers", "UCLA",
    "USC", "Washington", "Wisconsin", "Arizona", "Arizona State", "Baylor", "BYU", "Cincinnati",
    "Colorado", "Houston", "Iowa State", "Kansas", "Kansas State",
    "Oklahoma State", "TCU", "Texas Tech", "UCF", "Utah",
    "West Virginia", "Alabama", "Arkansas", "Auburn", "Florida", "Georgia",
    "Kentucky", "LSU", "Mississippi State", "Missouri", "Oklahoma",
    "Ole Miss", "South Carolina", "Tennessee", "Texas", "Texas A&M",
    "Vanderbilt", "Army", "Charlotte", "East Carolina", "Florida Atlantic", "Memphis",
    "Navy", "North Texas", "Rice", "Temple", "Tulane",
    "Tulsa", "UAB", "South Florida", "UTSA", "Boise State", "Colorado State", "Fresno State",
    "Oregon State", "San Diego State", "Texas State", "Utah State", "Washington State",
    "Air Force", "Hawai'i", "Nevada", "New Mexico", "North Dakota State",
    "Northern Illinois", "San José State", "UNLV", "UTEP", "Wyoming",
    "Akron", "Ball State", "Bowling Green", "Buffalo", "Central Michigan",
    "Eastern Michigan", "Kent State", "Miami (OH)", "Ohio", "Sacramento State",
    "Toledo", "Massachusetts", "Western Michigan",
    "Delaware", "Florida International", "Jacksonville State", "Kennesaw State", "Liberty",
    "Middle Tennessee", "Missouri State", "New Mexico State", "Sam Houston", "Western Kentucky",
    "Appalachian State", "Arkansas State", "Coastal Carolina", "Georgia Southern", "Georgia State",
    "James Madison", "Louisiana", "Louisiana Tech", "Marshall", "Old Dominion",
    "South Alabama", "Southern Miss", "Troy", "UL Monroe", "Notre Dame", "UConn"
]

# Now, let's filter the data so that it only keeps fbs vs fbs games
# and it only keeps the regular season games and plays that we care about on offense.
pbp_filtered = cfb_data[(cfb_data['homeTeamName'].isin(fbs_teams)) & (cfb_data['awayTeamName'].isin(fbs_teams))
].copy()

In [14]:
# Create downstrem filtering and on field next score calculation process
def process_next_score_data(df):
    # Base filtering
    filtered_1 = df[
        (df['seasonType'] == 2) &
        (df['EPA'].notna()) &
        (df['pos_team'].notna()) &
        (~df['type.text'].isin(['extra_point', 'Two Point Pass', 'Two Point Rush', 'Unknown', 'Defensive 2pt Conversion']))
    ].copy()

    # must filter out columns to conserve memory
    filtered = filtered_1[['game_id','half','fg_made','wp_before','touchdown','pos_team','rush_td','pass_td','homeTeamName', 'awayTeamName','def_pos_team', 'safety']].copy()

    # Step A: Identify home scoring events
    def calc_home_score_event(row):
        if row['touchdown'] == 1 and row['pos_team'] == row['homeTeamName'] and ((row['rush_td'] == True) or (row['pass_td'] == True)):
            return 7.0
        elif row['touchdown'] == 1 and row['pos_team'] == row['awayTeamName'] and ((row['rush_td'] == True) or (row['pass_td'] == True)):
            return -7.0
        elif (row['fg_made'] == True) and row['pos_team'] == row['homeTeamName']:
            return 3.0
        elif (row['fg_made'] == True) and row['pos_team'] == row['awayTeamName']:
            return -3.0
        elif (row['safety'] == True) and row['def_pos_team'] == row['homeTeamName']:
            return 2.0
        elif (row['safety'] == True) and row['def_pos_team'] == row['awayTeamName']:
            return -2.0
        return np.nan

    filtered['home_scoring_event'] = filtered.apply(calc_home_score_event, axis=1)

    # Step B: Backfill within game_id and game_half
    filtered['home_scoring_event'] = (
        filtered.groupby(['game_id', 'half'])['home_scoring_event']
        .bfill()
    )

    # Step C: Fill remaining missing with 0
    filtered['home_scoring_event'] = filtered['home_scoring_event'].fillna(0)

    # Step D: Convert to possession team perspective
    filtered['pts_next_score'] = np.where(
        filtered['pos_team'] == filtered['homeTeamName'],
        filtered['home_scoring_event'],
        -filtered['home_scoring_event']
    )

    # Apply garbage time filter
    filtered = filtered[
        (filtered['wp_before'].notna()) &
        (filtered['wp_before'] >= 0.05) &
        (filtered['wp_before'] <= 0.95)
    ].copy()

    return filtered

# Use the function on out data
pbp_filtered = process_next_score_data(pbp_filtered)

In [16]:
import matplotlib as plt

AttributeError: module 'matplotlib' has no attribute 'get_data_path'

In [15]:
def plot_calibration(df, bin_width=0.5, min_plays=50, title_suffix="2024 NFL Season"):
    df_copy = df.copy()
    df_copy['ep_bin'] = (df_copy['ep'] / bin_width).round() * bin_width

    cal_data = df_copy.groupby('ep_bin').agg(
        actual_pts=('pts_next_score', 'mean'),
        plays=('pts_next_score', 'count')
    ).reset_index()

    cal_data = cal_data[cal_data['plays'] >= min_plays]

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(
        data=cal_data, x='ep_bin', y='actual_pts', size='plays', 
        sizes=(30, 300), color='#013369', alpha=0.7, ax=ax
    )
    ax.plot([-4, 7], [-4, 7], linestyle='--', color='#D50A0A', linewidth=2, label='Perfect Calibration')
    
    ax.set_xticks(range(-4, 8))
    ax.set_yticks(range(-4, 8))
    ax.set_title(f"Expected Points Calibration ({title_suffix})", fontweight='bold', fontsize=14)
    ax.set_xlabel("Expected Points (Binned)")
    ax.set_ylabel("Average Actual Next Score")
    ax.legend(title="Number of Plays", loc="upper left")
    plt.tight_layout()
    plt.show()

plot_calibration(pbp_filtered)


KeyError: 'ep'